In [1]:
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt

# 1. Load dataset
df = pd.read_csv(r"C:\PROJECT\Probforecast\data\raw\PJME_hourly.csv")
df['Datetime'] = pd.to_datetime(df['Datetime'])
df = df.sort_values('Datetime').set_index('Datetime')

# 2. Handle duplicate timestamp telemetry artifacts
df = df.groupby(level=0).mean()

# 3. Feature Extraction Function
def create_time_features(data_frame):
    target_df = data_frame.copy()
    target_df['hour'] = target_df.index.hour
    target_df['dayofweek'] = target_df.index.dayofweek
    target_df['quarter'] = target_df.index.quarter
    target_df['month'] = target_df.index.month
    target_df['year'] = target_df.index.year
    target_df['dayofyear'] = target_df.index.dayofyear
    return target_df

df_features = create_time_features(df)

# 4. Chronological Splitting
cutoff_date = '2015-01-01'
train = df_features.loc[df_features.index < cutoff_date]
test = df_features.loc[df_features.index >= cutoff_date]

FEATURES = ['hour', 'dayofweek', 'quarter', 'month', 'year', 'dayofyear']
TARGET = 'PJME_MW'

X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]

print(f"Setup Complete! Training shape: {X_train.shape}, Testing shape: {X_test.shape}")

Setup Complete! Training shape: (113925, 6), Testing shape: (31437, 6)


In [2]:
# Initialize the XGBoost Regressor
reg = xgb.XGBRegressor(
    n_estimators=1000,
    early_stopping_rounds=50,
    learning_rate=0.01,
    max_depth=5,
    random_state=42
)

# Train the model using our split matrices
reg.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=100  # Prints out validation loss metrics every 100 iterations
)

[0]	validation_0-rmse:6409.88419	validation_1-rmse:6481.53202
[100]	validation_0-rmse:4056.43864	validation_1-rmse:4406.78824
[200]	validation_0-rmse:3395.56907	validation_1-rmse:3913.22698
[300]	validation_0-rmse:3145.68521	validation_1-rmse:3745.12732
[400]	validation_0-rmse:2980.08371	validation_1-rmse:3723.11757
[428]	validation_0-rmse:2947.46462	validation_1-rmse:3725.18135


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",50
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [3]:
# Extract the final loss metrics from the model's history
final_train_rmse = reg.evals_result()['validation_0']['rmse'][-1]
final_test_rmse = reg.evals_result()['validation_1']['rmse'][-1]

print("--- Final Baseline Performance ---")
print(f"Training Root Mean Squared Error: {final_train_rmse:.2f} MW")
print(f"Testing Root Mean Squared Error:  {final_test_rmse:.2f} MW")

--- Final Baseline Performance ---
Training Root Mean Squared Error: 2947.46 MW
Testing Root Mean Squared Error:  3725.18 MW
